# Euclidean ShellForce Corr Model on Continuous ABP

This notebook trains a short-time ShellForce CNEEP model on continuous-space WCA active Brownian particles.  Particle centers are converted to a fixed-grid Gaussian density field.  Orientation is intentionally hidden from the network so the learned signal corresponds to the apparent coarse-grained density-field irreversibility rather than particle-level active heat.  The model uses Euclidean annuli and `learned_absolute` shell messages, which is better matched to the radial WCA interaction than the Chebyshev shells used for lattice fields.

Training still uses only the coarse field pairs.  For evaluation, the saved particle positions and hidden orientations are used to compute the exact ABP medium entropy production, so the learned increments can be compared against a physical reference as well as WCA potential-energy diagnostics.

In [ ]:
import os
import sys

candidate_roots = [
    os.environ.get("CNEEP_V2_ROOT"),
    os.path.abspath(".."),
    os.path.abspath("."),
    "/home/user1/CNEEP_v2",
]

CNEEP_V2_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.exists(os.path.join(candidate, "data", "ABP", "core.py")):
        CNEEP_V2_ROOT = candidate
        break

if CNEEP_V2_ROOT is None:
    raise RuntimeError("Could not locate CNEEP_v2 root. Set CNEEP_V2_ROOT.")

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

print("CNEEP_v2 root:", CNEEP_V2_ROOT)

In [ ]:
from argparse import Namespace
from datetime import datetime
import math

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm import tqdm

from data.ABP import ABPParams, ContinuousABP, ABPFieldizer
from utils.sampler import CartesianSeqSampler

## 1. Hyperparameters

In [ ]:
opt = Namespace()
opt.model_type = "ABPEuclideanShellForceCNEEP2D_Tanh"
opt.device = "cuda" if torch.cuda.is_available() else "cpu"

# NEEP objective.  alpha=0 is the most transparent setting for checking
# whether the model sees a forward/reverse asymmetry at all.
opt.alpha = 0.0
opt.beta = 1.0
opt.lam = 0.0
opt.threshold = 0.01

# data/model shape
opt.periodic = True
opt.positional = False
opt.n_components = 1
opt.seq_len = 2

# Euclidean ShellForce settings
opt.max_distance = 5
opt.include_k0 = True
opt.shell_center_mode = "relative_only"
opt.shell_force_bias = False
opt.shell_relative_mode = "learned_absolute"
opt.shell_weight_normalization = "none"
opt.shell_width = 1.0
opt.shell_offset = 0.0
opt.shell_force_activation = "tanh"

# training
opt.n_iter = 3000
opt.train_batch_size = 256
opt.test_batch_size = 256
opt.video_batch_size = 256
opt.lr = 2e-4
opt.wd = 1e-6
opt.input_scalar = 1
opt.loss_scalar = 1
opt.scalar = 1
opt.clip_norm = 1
opt.record_freq = 100
opt.seed = 5
opt.n_layer = 2
opt.n_channel = 48
opt.n_hidden = 2
opt.val_ratio = 0.25

# Fixed-grid ABP simulation style shared with the steady-state sanity notebook.
# Change these four knobs first.  grid_size and dx fix the observation grid;
# changing target_phi changes N, not the box/grid.
target_phi = 0.60
grid_size = 32
dx = 2.0
wca_cutoff_pixels = 2.5

box_L = grid_size * dx
sigma = wca_cutoff_pixels * dx / (2.0 ** (1.0 / 6.0))
N_particles = max(1, int(round(4.0 * target_phi * box_L**2 / (math.pi * sigma**2))))

abp_params = ABPParams(
    N=N_particles,
    L=box_L,
    sigma=sigma,
    epsilon=0.5,
    mobility=1.0,
    force_clip=None,  # keep WCA conservative so the stored medium EP is exact
    force_chunk_size=65536,
    v0=10,
    Dr=1e-6,
    Dt=1e-6,
    dt=1e-4,
    seed=123,
    device=opt.device,
)

fieldizer = ABPFieldizer(
    box_size=abp_params.L,
    grid_size=grid_size,
    particle_diameter=abp_params.sigma,
    mode="gaussian",
    include_orientation=False,
    clip_occupancy=False,
    gaussian_sigma=0.5 * abp_params.sigma,
)
opt.n_components = fieldizer.n_channels
opt.input_shape = (grid_size, grid_size)

# Simulation controls.  Saved pairs are spaced like the steady-state sanity
# notebook, while exact medium EP is accumulated over all underlying integration
# steps inside each saved interval.
n_trajs = 6
n_trajs_test = 2
burn_in = 0
n_steps = 2_000_000
save_interval = 1000
n_steps_test = 2_000_000
sim_batch_size = 2
test_sim_batch_size = 1
test_seed = 456
dt_saved = abp_params.dt * save_interval

torch.manual_seed(opt.seed)
np.random.seed(opt.seed)

result_folder = os.path.join(CNEEP_V2_ROOT, "results")
current_result_folder = os.path.join(
    result_folder, f"CorrABP-EuclideanShellForce-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}"
)
os.makedirs(current_result_folder, exist_ok=True)
current_checkpoint_path = os.path.join(current_result_folder, "model_parameter.pth.tar")
best_checkpoint_path = os.path.join(current_result_folder, "best_model_parameter.pth.tar")

print(f"Device: {opt.device}")
print(f"Results: {current_result_folder}")
print(f"target_phi={target_phi:.3f}, actual_phi={abp_params.phi:.3f}, N={abp_params.N}")
print(f"ABP Pe={abp_params.Pe:.2f}, epsilon={abp_params.epsilon:.3f}, force_clip={abp_params.force_clip}")
print(f"Grid: {grid_size}x{grid_size}, dx={fieldizer.dx:.4f}, L={abp_params.L:.4f}")
print(f"sigma={abp_params.sigma:.4f} ({abp_params.sigma / fieldizer.dx:.3f} px)")
print(f"WCA rc={abp_params.rc:.4f} ({abp_params.rc / fieldizer.dx:.3f} px)")
print(f"Field channels: Gaussian density only  n_components={opt.n_components}, gaussian_sigma={fieldizer.gaussian_sigma:.4f}")
print(f"dt_saved={dt_saved:.4g}, active displacement per saved pair={abp_params.v0 * dt_saved / fieldizer.dx:.3f} pixels")
print(f"Trajectory counts: train={n_trajs}, test={n_trajs_test}, sim_batch_size={sim_batch_size}")
print("Training uses no explicit EP scale; evaluation uses exact medium EP accumulated during simulation.")
if abp_params.phi > 1.0:
    print("[WARN] phi > 1.0. This is a very soft-overlap/high-density WCA run, not a hard-disk-like packing.")
print(f"Euclidean shells: learned_absolute, max_distance={opt.max_distance}")

## 2. Simulate ABP trajectories and Gaussian fields

In [ ]:
def fields_to_video(fields):
    # fields: [T, B, C, H, W] -> [B, T, H, W] for density-only C=1.
    fields = fields.float()
    if fields.shape[2] != 1:
        raise ValueError(f"Expected density-only field, got {fields.shape[2]} channels.")
    return fields[:, :, 0].permute(1, 0, 2, 3).contiguous()


def concatenate_simulation_chunks(chunks):
    out = {"params": chunks[0]["params"], "times": chunks[0]["times"]}
    for key in [
        "positions",
        "theta",
        "fields",
        "potential",
        "min_distance",
        "mean_force_norm",
        "exact_active_medium_ep",
        "exact_wca_boundary_ep",
        "exact_medium_ep",
    ]:
        if key in chunks[0]:
            concat_dim = 0 if key.endswith("_ep") else 1
            out[key] = torch.cat([chunk[key] for chunk in chunks], dim=concat_dim)
    return out


def simulate_ensemble_in_batches(total_B, params, base_seed, batch_size, label, total_steps):
    chunks = []
    for start in range(0, total_B, batch_size):
        B = min(batch_size, total_B - start)
        batch_params = ABPParams(**{**params.__dict__, "seed": base_seed + start})
        sim = ContinuousABP(batch_params)
        print(f"[INFO] {label} chunk {start // batch_size + 1}: B={B}, seed={batch_params.seed}")
        chunks.append(
            sim.simulate(
                B=B,
                burn_in=burn_in,
                n_steps=total_steps,
                save_interval=save_interval,
                fieldizer=fieldizer,
                show_progress=True,
                save_exact_medium_ep=True,
            )
        )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return concatenate_simulation_chunks(chunks)


print(f"[INFO] Generating TRAIN ABP trajectories B={n_trajs}")
sim_train = ContinuousABP(abp_params)
result_train = simulate_ensemble_in_batches(
    n_trajs,
    abp_params,
    abp_params.seed or 0,
    sim_batch_size,
    "TRAIN",
    n_steps,
)
train_states = fields_to_video(result_train["fields"])

print(f"[INFO] Generating TEST ABP trajectories B={n_trajs_test}")
sim_test = ContinuousABP(ABPParams(**{**abp_params.__dict__, "seed": test_seed}))
result_test = simulate_ensemble_in_batches(
    n_trajs_test,
    ABPParams(**{**abp_params.__dict__, "seed": test_seed}),
    test_seed,
    test_sim_batch_size,
    "TEST",
    n_steps_test,
)
test_states = fields_to_video(result_test["fields"])

print("Train field states:", train_states.shape)
print("Test field states: ", test_states.shape)
print("Final field diagnostics:", fieldizer.diagnostics_dict(result_train["positions"][-1].to(opt.device)))

## 3. Prepare tensors and normalization

In [ ]:
opt.M = train_states.shape[0]
opt.L = train_states.shape[1]
opt.M_test = test_states.shape[0]
opt.L_test = test_states.shape[1]

train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
M_train_new = train_val_split_idx
M_val = opt.M - M_train_new

# Keep full videos on CPU; sampled batches move to opt.device inside the
# training/evaluation loops.
train_video = train_states[:M_train_new].float()
val_video = train_states[M_train_new:].float()
test_video = test_states.float()

mean = torch.mean(train_video, dim=(0, 1, 2, 3), keepdim=True)
std = torch.std(train_video, dim=(0, 1, 2, 3), keepdim=True).clamp_min(1e-6)
transform = lambda x: (x - mean.to(x.device)) * opt.input_scalar / std.to(x.device)

train_U = result_train["potential"].numpy().T
test_U = result_test["potential"].numpy().T
test_min_dist = result_test["min_distance"].numpy().T
test_times = result_test["times"].numpy()
train_exact_medium_ep = result_train["exact_medium_ep"].numpy()
test_exact_active_ep = result_test["exact_active_medium_ep"].numpy()
test_exact_wca_boundary_ep = result_test["exact_wca_boundary_ep"].numpy()
test_exact_medium_ep = result_test["exact_medium_ep"].numpy()

print("Train video:", train_video.shape)
print("Val video:  ", val_video.shape)
print("Test video: ", test_video.shape)
print(f"Video tensors stored on {train_video.device}; sampled batches move to {opt.device}.")
print("Density mean:", float(mean.detach().cpu()))
print("Density std: ", float(std.detach().cpu()))
print("Exact medium EP grid:", test_exact_medium_ep.shape)
print("Mean exact medium EP rate train/test:", float(train_exact_medium_ep.mean() / dt_saved), float(test_exact_medium_ep.mean() / dt_saved))
print("Mean WCA U train/test:", float(train_U.mean()), float(test_U.mean()))

## 4. Quick data sanity plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, t in enumerate([0, opt.L // 2, opt.L - 1]):
    im = axes[0, col].imshow(train_video[0, t].detach().cpu().numpy().T, origin="lower", cmap="viridis")
    axes[0, col].set_title(f"train Gaussian field t={t}")
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

axes[1, 0].plot(result_train["times"].numpy(), train_U.T, alpha=0.6)
axes[1, 0].set_title("WCA potential")
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("U")

axes[1, 1].plot(result_train["times"].numpy(), result_train["min_distance"].numpy() / abp_params.sigma, alpha=0.6)
axes[1, 1].axhline(1.0, color="k", linestyle="--", lw=1)
axes[1, 1].set_title("min distance / sigma")
axes[1, 1].set_xlabel("time")

axes[1, 2].hist(train_U.reshape(-1), bins=40, alpha=0.8)
axes[1, 2].set_title("WCA U distribution")
axes[1, 2].set_xlabel("U")

plt.tight_layout()
plt.show()

## 5. Build Euclidean ShellForce model

In [ ]:
from models.NEEP_ABP_ShellForce_2D import ABPEuclideanShellForceCNEEP2D, ABPEuclideanShellForceCNEEP2D_Tanh

if opt.model_type == "ABPEuclideanShellForceCNEEP2D_Tanh":
    model = ABPEuclideanShellForceCNEEP2D_Tanh(opt).to(opt.device)
elif opt.model_type == "ABPEuclideanShellForceCNEEP2D":
    model = ABPEuclideanShellForceCNEEP2D(opt).to(opt.device)
else:
    raise ValueError(f"Unknown model_type: {opt.model_type}")

optim = torch.optim.AdamW(model.parameters(), opt.lr, weight_decay=opt.wd)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print("annulus bounds in pixels:", model.shell_bounds())
print("annulus bounds in sigma units:", [(a * fieldizer.dx / abp_params.sigma, b * fieldizer.dx / abp_params.sigma) for a, b in model.shell_bounds()])

## 6. Train

In [ ]:
train_sampler = CartesianSeqSampler(
    M_train_new, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device
)
val_sampler = CartesianSeqSampler(
    M_val, opt.L, opt.seq_len, opt.test_batch_size, device=opt.device, train=False
)

best_val_loss = float("inf")
smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None
history_train_loss = []
history_val_loss = []
history_iters = []

plt.ion()
fig, ax = plt.subplots(figsize=(8, 5))
line_train, = ax.plot([], [], label="Train Loss")
line_val, = ax.plot([], [], label="Val Loss")
ax.set_xlabel("Iteration")
ax.set_ylabel("Loss")
ax.set_title("Training and Validation Loss")
ax.legend()
ax.grid(True)
display_handle = display(fig, display_id=True)
plt.close(fig)

for it in tqdm(range(1, opt.n_iter + 1)):
    model.train()
    batch = next(train_sampler)
    b0 = batch[0].to(train_video.device)
    slices = [train_video[(b0, batch[1][i].to(train_video.device))] for i in range(opt.seq_len)]
    x = transform(torch.stack(slices, dim=1).float().to(opt.device))

    J_all = model(x) / opt.scalar
    ep_density = J_all.sum(dim=1)

    optim.zero_grad()
    if opt.alpha == 0:
        loss = (-ep_density + (torch.exp(-ep_density) - 1)).mean()
    else:
        loss = (
            -(torch.exp(opt.alpha * ep_density) - 1) / opt.alpha
            + (torch.exp(-(1 + opt.alpha) * ep_density) - 1) / (1 + opt.alpha)
        ).mean()

    (loss * opt.loss_scalar).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=opt.clip_norm)
    optim.step()

    if it % opt.record_freq == 0 or it == 1:
        model.eval()
        val_loss_acc = 0.0
        n_val = 0
        with torch.no_grad():
            for vb in val_sampler:
                vb0 = vb[0].to(val_video.device)
                vslices = [val_video[(vb0, vb[1][i].to(val_video.device))] for i in range(opt.seq_len)]
                vx = transform(torch.stack(vslices, dim=1).float().to(opt.device))
                vJ = model(vx) / opt.scalar
                v_ep_density = vJ.sum(dim=1)
                if opt.alpha == 0:
                    vloss = (-v_ep_density + (torch.exp(-v_ep_density) - 1)).sum().item()
                else:
                    vloss = (
                        -(torch.exp(opt.alpha * v_ep_density) - 1) / opt.alpha
                        + (torch.exp(-(1 + opt.alpha) * v_ep_density) - 1) / (1 + opt.alpha)
                    ).sum().item()
                val_loss_acc += vloss
                n_val += vx.shape[0]

        avg_val = val_loss_acc / max(n_val, 1)
        state = {
            "settings": opt.__dict__,
            "state_dict": model.state_dict(),
            "optimizer": optim.state_dict(),
            "iteration": it,
            "channel_mean": mean.detach().cpu(),
            "channel_std": std.detach().cpu(),
            "abp_params": abp_params.__dict__,
            "fieldizer": {
                "box_size": fieldizer.box_size,
                "grid_size": fieldizer.grid_size,
                "particle_diameter": fieldizer.particle_diameter,
                "mode": fieldizer.mode,
                "include_orientation": fieldizer.include_orientation,
                "clip_occupancy": fieldizer.clip_occupancy,
                "gaussian_sigma": fieldizer.gaussian_sigma,
                "gaussian_sigma_pixels": fieldizer.gaussian_sigma_pixels,
                "gaussian_truncate": fieldizer.gaussian_truncate,
                "gaussian_normalize": fieldizer.gaussian_normalize,
            },
            "simulation_style": {
                "target_phi": target_phi,
                "actual_phi": abp_params.phi,
                "grid_size": grid_size,
                "dx": dx,
                "wca_cutoff_pixels": wca_cutoff_pixels,
                "n_particles": abp_params.N,
                "save_exact_medium_ep": True,
            },
            "dt_saved": dt_saved,
        }
        torch.save(state, current_checkpoint_path)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(state, best_checkpoint_path)

        smooth_train_loss = loss.item() if smooth_train_loss is None else smoothing * smooth_train_loss + (1 - smoothing) * loss.item()
        smooth_val_loss = avg_val if smooth_val_loss is None else smoothing * smooth_val_loss + (1 - smoothing) * avg_val
        history_train_loss.append(smooth_train_loss)
        history_val_loss.append(smooth_val_loss)
        history_iters.append(it)
        line_train.set_data(history_iters, history_train_loss)
        line_val.set_data(history_iters, history_val_loss)
        ax.relim()
        ax.autoscale_view()
        display_handle.update(fig)

print("Training finished.")
print(f"Best checkpoint: {best_checkpoint_path}")

## 7. Load best or final model

In [ ]:
load_best = True
checkpoint_path = best_checkpoint_path if load_best and os.path.exists(best_checkpoint_path) else current_checkpoint_path
checkpoint = torch.load(checkpoint_path, map_location=opt.device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"Loaded {checkpoint_path}")
print(f"Iteration: {checkpoint.get('iteration', 'unknown')}")

## 8. Predict test-pair EP-like increments

In [ ]:
model.eval()
pred_increment = []
pred_shell_increment = []
pair_b = []
pair_t = []

test_sampler = CartesianSeqSampler(
    opt.M_test, opt.L_test, opt.seq_len, opt.video_batch_size, device=opt.device, train=False
)

with torch.no_grad():
    for batch in tqdm(test_sampler):
        b0 = batch[0].to(test_video.device)
        t0 = batch[1][0].to(test_video.device)
        slices = [test_video[(b0, batch[1][i].to(test_video.device))] for i in range(opt.seq_len)]
        x = transform(torch.stack(slices, dim=1).float().to(opt.device))
        J_all = model(x) / opt.scalar
        shell_inc = J_all.detach().cpu().numpy()
        total_inc = shell_inc.sum(axis=1)
        pred_shell_increment.append(shell_inc)
        pred_increment.append(total_inc)
        pair_b.append(batch[0].detach().cpu().numpy())
        pair_t.append(batch[1][0].detach().cpu().numpy())

pred_increment = np.concatenate(pred_increment)
pred_shell_increment = np.concatenate(pred_shell_increment, axis=0)
pair_b = np.concatenate(pair_b)
pair_t = np.concatenate(pair_t)

U_t = test_U[pair_b, pair_t]
U_tp1 = test_U[pair_b, pair_t + 1]
dU_dt = (U_tp1 - U_t) / dt_saved
release_rate = -dU_dt
min_r = test_min_dist[pair_b, pair_t]
pred_rate = pred_increment / dt_saved
shell_rate = pred_shell_increment / dt_saved
pair_time = test_times[pair_t]

def corrcoef(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return float(np.corrcoef(x[mask], y[mask])[0, 1])

print(f"Pred mean rate: {pred_rate.mean():.6e}")
print(f"corr(pred, U):        {corrcoef(pred_rate, U_t):.4f}")
print(f"corr(pred, -dU/dt):   {corrcoef(pred_rate, release_rate):.4f}")
print(f"corr(pred, min_dist): {corrcoef(pred_rate, min_r):.4f}")

## Exact Medium Entropy Production Comparison

In [ ]:
# Exact ABP medium EP on the same test pairs used for model prediction.
# These arrays were accumulated during simulation.  The active-work term is
# summed over every integration step inside each saved interval, so this remains
# exact even when save_interval > 1.
required_ep_keys = ["exact_active_medium_ep", "exact_wca_boundary_ep", "exact_medium_ep"]
missing_ep_keys = [key for key in required_ep_keys if key not in result_test]
if missing_ep_keys:
    raise KeyError(f"Missing exact EP arrays from result_test: {missing_ep_keys}")
if abp_params.force_clip is not None:
    print(
        "[WARN] force_clip is enabled. The WCA boundary term uses the stored WCA "
        "potential, while clipped forces are used in the simulated drift."
    )

exact_active_inc_grid = result_test["exact_active_medium_ep"].numpy()
wca_boundary_inc_grid = result_test["exact_wca_boundary_ep"].numpy()
exact_medium_inc_grid = result_test["exact_medium_ep"].numpy()

exact_active_increment = exact_active_inc_grid[pair_b, pair_t]
exact_medium_increment = exact_medium_inc_grid[pair_b, pair_t]
wca_boundary_increment = wca_boundary_inc_grid[pair_b, pair_t]

exact_active_rate = exact_active_increment / dt_saved
exact_medium_rate = exact_medium_increment / dt_saved
wca_boundary_rate = wca_boundary_increment / dt_saved

def finite_pair_mask(*arrays):
    mask = np.ones_like(np.asarray(arrays[0], dtype=float), dtype=bool)
    for arr in arrays:
        mask &= np.isfinite(arr)
    return mask

def best_scale(source, target):
    mask = finite_pair_mask(source, target)
    denom = float(np.dot(source[mask], source[mask]))
    if denom <= 1e-30 or mask.sum() < 3:
        return np.nan
    return float(np.dot(source[mask], target[mask]) / denom)

def r2_score(y_true, y_pred):
    mask = finite_pair_mask(y_true, y_pred)
    if mask.sum() < 3:
        return np.nan
    y = y_true[mask]
    yp = y_pred[mask]
    denom = float(np.sum((y - y.mean()) ** 2))
    if denom <= 1e-30:
        return np.nan
    return float(1.0 - np.sum((y - yp) ** 2) / denom)

scale_pred_to_medium = best_scale(pred_increment, exact_medium_increment)
scale_pred_to_active = best_scale(pred_increment, exact_active_increment)
cal_pred_medium_increment = scale_pred_to_medium * pred_increment
cal_pred_medium_rate = cal_pred_medium_increment / dt_saved

print(f"Exact active-work mean rate: {exact_active_rate.mean():.6e}")
print(f"WCA boundary mean rate:      {wca_boundary_rate.mean():.6e}")
print(f"Exact medium mean rate:      {exact_medium_rate.mean():.6e}")
print(f"Pred mean rate:              {pred_rate.mean():.6e}")
print(f"corr(pred, exact active):    {corrcoef(pred_rate, exact_active_rate):.4f}")
print(f"corr(pred, exact medium):    {corrcoef(pred_rate, exact_medium_rate):.4f}")
print(f"corr(pred, WCA boundary):    {corrcoef(pred_rate, wca_boundary_rate):.4f}")
print(f"scale pred -> exact active:  {scale_pred_to_active:.6e}")
print(f"scale pred -> exact medium:  {scale_pred_to_medium:.6e}")
print(f"R2 scaled pred vs medium:    {r2_score(exact_medium_increment, cal_pred_medium_increment):.4f}")

# Reconstruct exact/predicted increments on the [ensemble, time-pair] grid.
pred_inc_grid_for_exact = np.full((opt.M_test, opt.L_test - 1), np.nan, dtype=np.float64)
for inc, b, t in zip(pred_increment, pair_b, pair_t):
    if 0 <= b < opt.M_test and 0 <= t < opt.L_test - 1:
        pred_inc_grid_for_exact[b, t] = inc
missing = np.isnan(pred_inc_grid_for_exact)
if missing.any():
    print(f"[WARN] Missing predicted increments in {missing.sum()} pair slots; filling them with 0 for cumulative EP plots.")
pred_inc_grid_for_exact = np.nan_to_num(pred_inc_grid_for_exact, nan=0.0)

ens = 0
time_pairs = test_times[1:]
cum_exact_medium = np.cumsum(exact_medium_inc_grid[ens])
cum_exact_active = np.cumsum(exact_active_inc_grid[ens])
cum_pred = np.cumsum(pred_inc_grid_for_exact[ens])
cum_pred_scaled = scale_pred_to_medium * cum_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].scatter(exact_medium_rate, pred_rate, s=8, alpha=0.35)
axes[0, 0].set_xlabel("exact medium EP rate")
axes[0, 0].set_ylabel("predicted rate")
axes[0, 0].set_title(f"raw scale, corr={corrcoef(pred_rate, exact_medium_rate):.3f}")

axes[0, 1].scatter(exact_medium_rate, cal_pred_medium_rate, s=8, alpha=0.35, color="tab:green")
lo = np.nanpercentile(exact_medium_rate, 1)
hi = np.nanpercentile(exact_medium_rate, 99)
axes[0, 1].plot([lo, hi], [lo, hi], color="k", linestyle="--", lw=1)
axes[0, 1].set_xlabel("exact medium EP rate")
axes[0, 1].set_ylabel("scaled predicted rate")
axes[0, 1].set_title(f"scale-only fit, R2={r2_score(exact_medium_increment, cal_pred_medium_increment):.3f}")

axes[1, 0].plot(time_pairs, cum_exact_medium, label="exact medium EP", lw=2)
axes[1, 0].plot(time_pairs, cum_exact_active, label="exact active-work EP", lw=1.5, alpha=0.8)
axes[1, 0].plot(time_pairs, cum_pred_scaled, label="scaled cumulative pred", lw=2, linestyle="--")
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("cumulative increment")
axes[1, 0].set_title(f"ensemble {ens}: cumulative comparison")
axes[1, 0].legend()

axes[1, 1].hist(exact_medium_rate, bins=60, alpha=0.65, label="exact medium")
axes[1, 1].hist(cal_pred_medium_rate, bins=60, alpha=0.55, label="scaled pred")
axes[1, 1].set_xlabel("EP rate")
axes[1, 1].set_ylabel("count")
axes[1, 1].set_title("Rate distributions")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_exact_medium_ep_comparison.png"), dpi=150)
plt.show()

## 9. Endpoint WCA Potential Budget

In [ ]:
# Reconstruct predicted increments on the [ensemble, time-pair] grid.
pred_inc_grid = np.full((opt.M_test, opt.L_test - 1), np.nan, dtype=np.float64)
for inc, b, t in zip(pred_increment, pair_b, pair_t):
    if 0 <= b < opt.M_test and 0 <= t < opt.L_test - 1:
        pred_inc_grid[b, t] = inc

missing = np.isnan(pred_inc_grid)
if missing.any():
    print(f"[WARN] Missing predicted increments in {missing.sum()} pair slots; filling them with 0 for cumulative plots.")
pred_inc_grid = np.nan_to_num(pred_inc_grid, nan=0.0)

cum_pred_ep = np.cumsum(pred_inc_grid, axis=1)
minus_delta_U_time = -(test_U[:, 1:] - test_U[:, [0]])

endpoint_rows = []
for b in range(opt.M_test):
    U_first = float(test_U[b, 0])
    U_last = float(test_U[b, -1])
    delta_U = U_last - U_first
    cumulative_pred_ep = float(cum_pred_ep[b, -1])
    endpoint_rows.append((f"ens {b}", U_first, U_last, delta_U, -delta_U, cumulative_pred_ep))

total_U = test_U.sum(axis=0)
total_delta_U = float(total_U[-1] - total_U[0])
total_cum_pred_ep = float(cum_pred_ep[:, -1].sum())
endpoint_rows.append(("TOTAL", float(total_U[0]), float(total_U[-1]), total_delta_U, -total_delta_U, total_cum_pred_ep))

print("label    | U(first) | U(last) | Delta U | -Delta U | cumulative predicted EP")
for row in endpoint_rows:
    print(
        f"{row[0]:>7s} | {row[1]:9.4e} | {row[2]:8.4e} | "
        f"{row[3]:+8.4e} | {row[4]:+8.4e} | {row[5]:+12.4e}"
    )

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for b in range(opt.M_test):
    axes[0, 0].plot(test_times, test_U[b], alpha=0.65, label=f"ens {b}")
    axes[0, 0].scatter([test_times[0], test_times[-1]], [test_U[b, 0], test_U[b, -1]], s=35)
axes[0, 0].plot(test_times, total_U, color="black", lw=2.2, label="TOTAL")
axes[0, 0].scatter([test_times[0], test_times[-1]], [total_U[0], total_U[-1]], color="black", s=50)
axes[0, 0].set_xlabel("time")
axes[0, 0].set_ylabel("WCA U")
axes[0, 0].set_title("WCA potential: ensemble and total")
axes[0, 0].legend()

labels = [row[0] for row in endpoint_rows]
xpos = np.arange(len(endpoint_rows))
minus_delta_U = np.array([row[4] for row in endpoint_rows])
endpoint_pred_ep = np.array([row[5] for row in endpoint_rows])
width = 0.38
axes[0, 1].bar(xpos - width / 2, minus_delta_U, width, label="-Delta U")
axes[0, 1].bar(xpos + width / 2, endpoint_pred_ep, width, label="cum. pred EP")
axes[0, 1].axhline(0, color="k", lw=1)
axes[0, 1].set_xticks(xpos)
axes[0, 1].set_xticklabels(labels, rotation=20)
axes[0, 1].set_title("Endpoint budget including TOTAL")
axes[0, 1].legend()

time_pairs = test_times[1:]
for b in range(opt.M_test):
    axes[1, 0].plot(time_pairs, minus_delta_U_time[b], alpha=0.65, label=f"-Delta U ens {b}")
    axes[1, 0].plot(time_pairs, cum_pred_ep[b], alpha=0.65, linestyle="--", label=f"cum EP ens {b}")
axes[1, 0].axhline(0, color="k", lw=1)
axes[1, 0].set_xlabel("time")
axes[1, 0].set_ylabel("cumulative value")
axes[1, 0].set_title("Per-ensemble cumulative EP vs potential boundary term")
axes[1, 0].legend(ncol=2, fontsize=8)

total_minus_delta_U_time = -(total_U[1:] - total_U[0])
total_cum_pred_ep_time = cum_pred_ep.sum(axis=0)
axes[1, 1].plot(time_pairs, total_minus_delta_U_time, label="TOTAL -Delta U", lw=2)
axes[1, 1].plot(time_pairs, total_cum_pred_ep_time, label="TOTAL cumulative pred EP", lw=2)
axes[1, 1].axhline(0, color="k", lw=1)
axes[1, 1].set_xlabel("time")
axes[1, 1].set_ylabel("cumulative value")
axes[1, 1].set_title("TOTAL cumulative comparison")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "endpoint_potential_budget.png"), dpi=150)
plt.show()

## 10. Trend plots against WCA observables

In [ ]:
def smooth(x, window=11):
    window = min(window, len(x))
    if window <= 1:
        return x
    return np.convolve(x, np.ones(window) / window, mode="same")

ens = 0
mask = pair_b == ens
order = np.argsort(pair_t[mask])
t_plot = pair_time[mask][order]
pred_plot = pred_rate[mask][order]
U_plot = U_t[mask][order]
rel_plot = release_rate[mask][order]
minr_plot = min_r[mask][order] / abp_params.sigma
pair_t_plot = pair_t[mask][order]
cum_pred_plot = np.cumsum(pred_increment[mask][order])
minus_delta_plot = -(test_U[ens, pair_t_plot + 1] - test_U[ens, 0])

fig, axes = plt.subplots(5, 1, figsize=(12, 12), sharex=True)
axes[0].plot(t_plot, pred_plot, alpha=0.35)
axes[0].plot(t_plot, smooth(pred_plot), lw=2)
axes[0].set_ylabel("Pred EP rate")

axes[1].plot(t_plot, cum_pred_plot, label="cum. pred EP", lw=2)
axes[1].plot(t_plot, minus_delta_plot, label="-Delta U from first", lw=2, alpha=0.8)
axes[1].axhline(0, color="k", lw=1)
axes[1].set_ylabel("cumulative")
axes[1].legend()

axes[2].plot(t_plot, U_plot, alpha=0.35, color="tab:orange")
axes[2].plot(t_plot, smooth(U_plot), lw=2, color="tab:orange")
axes[2].set_ylabel("WCA U")

axes[3].plot(t_plot, rel_plot, alpha=0.35, color="tab:green")
axes[3].plot(t_plot, smooth(rel_plot), lw=2, color="tab:green")
axes[3].axhline(0, color="k", lw=1)
axes[3].set_ylabel("-dU/dt")

axes[4].plot(t_plot, minr_plot, alpha=0.35, color="tab:red")
axes[4].plot(t_plot, smooth(minr_plot), lw=2, color="tab:red")
axes[4].axhline(1.0, color="k", linestyle="--", lw=1)
axes[4].set_ylabel("min r/sigma")
axes[4].set_xlabel("time")

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_wca_trends.png"), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(U_t, pred_rate, s=8, alpha=0.35)
axes[0].set_xlabel("WCA U(t)")
axes[0].set_ylabel("Pred EP rate")
axes[0].set_title(f"corr={corrcoef(pred_rate, U_t):.3f}")

axes[1].scatter(release_rate, pred_rate, s=8, alpha=0.35)
axes[1].set_xlabel("-dU/dt")
axes[1].set_ylabel("Pred EP rate")
axes[1].set_title(f"corr={corrcoef(pred_rate, release_rate):.3f}")

axes[2].scatter(min_r / abp_params.sigma, pred_rate, s=8, alpha=0.35)
axes[2].axvline(1.0, color="k", linestyle="--", lw=1)
axes[2].set_xlabel("min pair distance / sigma")
axes[2].set_ylabel("Pred EP rate")
axes[2].set_title(f"corr={corrcoef(pred_rate, min_r):.3f}")

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_wca_scatter.png"), dpi=150)
plt.show()

## 11. Local predicted map vs center WCA-energy field

In [ ]:
def particle_wca_energy(pos_np, sim):
    pos = torch.as_tensor(pos_np, dtype=torch.float32, device=opt.device).unsqueeze(0)
    p = sim.params
    delta = pos[:, :, None, :] - pos[:, None, :, :]
    delta = delta - p.L * torch.round(delta / p.L)
    r2 = torch.sum(delta * delta, dim=-1)
    eye = torch.eye(pos.shape[1], device=pos.device, dtype=torch.bool).unsqueeze(0)
    mask = (r2 > 0) & (r2 < p.rc ** 2) & (~eye)
    r2_safe = torch.where(mask, r2, torch.ones_like(r2))
    sig2_over_r2 = (p.sigma ** 2) / r2_safe
    sig6 = sig2_over_r2 ** 3
    sig12 = sig6 ** 2
    u_pair = 4.0 * p.epsilon * (sig12 - sig6) + p.epsilon
    u_pair = torch.where(mask, u_pair, torch.zeros_like(u_pair))
    return (0.5 * u_pair.sum(dim=2))[0].detach().cpu().numpy()


def particle_values_to_center_field(pos_np, values, fieldizer):
    H = fieldizer.grid_size
    dx = fieldizer.dx
    idx = np.floor((pos_np % fieldizer.box_size) / dx).astype(np.int64) % H
    lin = idx[:, 0] * H + idx[:, 1]
    out = np.zeros(H * H, dtype=np.float64)
    np.add.at(out, lin, values)
    return out.reshape(H, H)


b = 0
t = int(opt.L_test * 0.6)
x = transform(torch.stack([test_video[b:b+1, t], test_video[b:b+1, t + 1]], dim=1).float().to(opt.device))
with torch.no_grad():
    maps = model(x, return_maps=True) / opt.scalar
pred_map_rate = maps[0].sum(dim=0).detach().cpu().numpy() / dt_saved

pos_np = result_test["positions"][t, b].numpy()
u_particle = particle_wca_energy(pos_np, sim_test)
wca_field = particle_values_to_center_field(pos_np, u_particle, fieldizer)
occ = test_video[b, t].detach().cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(occ.T, origin="lower", cmap="viridis")
axes[0].set_title("Gaussian density field")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

v = np.nanpercentile(np.abs(pred_map_rate), 99)
v = max(v, 1e-12)
im1 = axes[1].imshow(pred_map_rate.T, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
axes[1].set_title("predicted local EP rate")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(wca_field.T, origin="lower", cmap="magma")
axes[2].set_title("center WCA energy field")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_local_map_vs_wca.png"), dpi=150)
plt.show()

print("Frame WCA U from particles:", float(u_particle.sum()))
print("Frame WCA U from diagnostics:", float(test_U[b, t]))

## 12. Shell spectrum on Euclidean radial bins

In [ ]:
mean_shell = shell_rate.mean(axis=0)
stderr_shell = shell_rate.std(axis=0) / np.sqrt(max(len(shell_rate), 1))
bounds_px = model.shell_bounds()
centers_sigma = np.array([0.0 if b == 0 else 0.5 * (lo + hi) * fieldizer.dx / abp_params.sigma for b, (lo, hi) in enumerate(bounds_px)])
labels = [str(i) for i in range(len(bounds_px))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(centers_sigma, mean_shell, yerr=stderr_shell, width=0.55 * fieldizer.dx / abp_params.sigma, capsize=3, alpha=0.8)
axes[0].axvline(abp_params.rc / abp_params.sigma, color="k", linestyle="--", label="WCA cutoff")
axes[0].set_xlabel("annulus center radius / sigma")
axes[0].set_ylabel("mean predicted shell rate")
axes[0].set_title("Euclidean ShellForce spectrum")
axes[0].legend()

axes[1].plot(centers_sigma, np.cumsum(mean_shell), "o-")
axes[1].axvline(abp_params.rc / abp_params.sigma, color="k", linestyle="--", label="WCA cutoff")
axes[1].set_xlabel("annulus center radius / sigma")
axes[1].set_ylabel("cumulative predicted rate")
axes[1].set_title("Cumulative shell contribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(current_result_folder, "abp_shell_spectrum.png"), dpi=150)
plt.show()

for idx, (lo, hi) in enumerate(bounds_px):
    print(
        f"branch {idx:02d}: pixels=({lo:.2f}, {hi:.2f}], "
        f"sigma_units=({lo * fieldizer.dx / abp_params.sigma:.3f}, {hi * fieldizer.dx / abp_params.sigma:.3f}], "
        f"mean_rate={mean_shell[idx]:.6e}"
    )

## 13. Inspect Euclidean annulus masks

In [ ]:
branches = [branch for branch in model.branches if branch.k > 0]
n_show = min(4, len(branches))
fig, axes = plt.subplots(1, n_show, figsize=(3.3 * n_show, 3.2))
if n_show == 1:
    axes = [axes]
for ax, branch in zip(axes, branches[:n_show]):
    offsets = branch.rel_proj.offsets.detach().cpu().numpy()
    ax.scatter(offsets[:, 1], offsets[:, 0], s=45)
    circle_outer = plt.Circle((0, 0), branch.r_outer, fill=False, color="k", linestyle="--")
    circle_inner = plt.Circle((0, 0), branch.r_inner, fill=False, color="gray", linestyle=":")
    ax.add_patch(circle_outer)
    ax.add_patch(circle_inner)
    ax.axhline(0, color="0.8", lw=1)
    ax.axvline(0, color="0.8", lw=1)
    ax.set_aspect("equal")
    ax.set_title(f"k={branch.k}: ({branch.r_inner:.1f},{branch.r_outer:.1f}] px")
plt.tight_layout()
plt.show()